In [1]:
%pip install pandas

In [2]:
%pip install numpy
%pip install seaborn

In [3]:
%pip install scikit-learn

In [4]:
%pip install matplotlib

In [5]:
%pip install tensorflow


In [7]:
from pathlib import Path

# 👉 Change this to your root images folder
root_dir = Path(r"/Users/dipak/Downloads/archive (10)/images")

# Which file extensions to treat as images
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

class_counts = {}
total = 0

# Assume structure: root_dir / <class_name> / (maybe subfolders) / image files
for class_dir in sorted(d for d in root_dir.iterdir() if d.is_dir()):
    class_name = class_dir.name

    count = 0
    # Walk recursively inside each class directory
    for path in class_dir.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
            count += 1

    class_counts[class_name] = count
    total += count

# Print per-class counts
for cls, cnt in class_counts.items():
    print(f"{cls:20s} : {cnt}")

print("-" * 40)
print(f"TOTAL IMAGES       : {total}")


apple_pie            : 1000
baby_back_ribs       : 1000
baklava              : 1000
beef_carpaccio       : 1000
beef_tartare         : 1000
beet_salad           : 1000
beignets             : 1000
bibimbap             : 1000
bread_pudding        : 1000
breakfast_burrito    : 1000
bruschetta           : 1000
caesar_salad         : 1000
cannoli              : 1000
caprese_salad        : 1000
carrot_cake          : 1000
ceviche              : 1000
cheese_plate         : 1000
cheesecake           : 1000
chicken_curry        : 1000
chicken_quesadilla   : 1000
chicken_wings        : 1000
chocolate_cake       : 1000
chocolate_mousse     : 1000
churros              : 1000
clam_chowder         : 1000
club_sandwich        : 1000
crab_cakes           : 1000
creme_brulee         : 1000
croque_madame        : 1000
cup_cakes            : 1000
deviled_eggs         : 1000
donuts               : 1000
dumplings            : 1000
edamame              : 1000
eggs_benedict        : 1000
escargots           

In [7]:
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf

# 1. GPU SETUP
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  #detects gpu
        DEVICE = '/GPU:0'
        print("✅ Using device:", DEVICE)
    except RuntimeError as e:
        print("GPU config error:", e)
        DEVICE = '/CPU:0'
else:
    print("⚠️ No GPU detected. Training will run on CPU.")
    DEVICE = '/CPU:0'

# 2. DATA PREP
image_dir = Path(r'C:\Users\dkushwaha\Downloads\archive\images')  #path of our image directory
filepaths = list(image_dir.glob('**/*.jpg'))  #any sub folder- any file ending in .jpg
labels = [os.path.split(os.path.split(str(x))[0])[1] for x in filepaths]  #gives the class label fo each image which is the parent directory name
df = pd.DataFrame({'Filepath': list(map(str, filepaths)), 'Label': labels})
'''creates a dataframe with two columns, one column is filepath: full path to each iamge as string and the other is label: class name for that image'''

num_classes = df['Label'].nunique() #counts how many distinct classes
print(f"Found {num_classes} classes")

category_samples = []  #class balancing - capping at 200 pr class here we create an empty list that will eventually store one small DF per cateory 
#each containing upto 200 rows of images for that class
for category in df['Label'].unique():
    cat_slice = df[df['Label'] == category] #filtering for label as current cateogory
    n_samples = min(200, len(cat_slice)) #taking only 200 sample from the total number of sample in that particular class
    if n_samples > 0:  #only when there is at least a record in the data frame
        category_samples.append(cat_slice.sample(n_samples, random_state=1)) #randomly pickes n_samples rows from cat_slice
        #random_state=1 sets the random seed so results are reproducible
df_balanced = pd.concat(category_samples).sample(frac=1.0, random_state=1).reset_index(drop=True)
#df_balanced is a balanced-ish dataset where no class has more than 200 images

train_df, test_df = train_test_split(df_balanced, train_size=0.7, shuffle=True, random_state=42)

# 3. DATA GENERATORS

'''
In deep learning with images, you usually don’t:

load all images into RAM yourself, and

manually resize/normalize/augment them.

Instead, you use a generator that:

Reads images from disk in small batches.

Resizes them to the input size (224×224 here).

Applies preprocessing (like ResNet normalization).

Optionally applies data augmentation (random transformations).

Yields (batch_images, batch_labels) to the model during training.

That’s what ImageDataGenerator does.'''
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    #the below function applies the exact preprocessing that restnet50 expects, converts image from RGB to BGR
    #subtract image net mena valies
    #resnet 50 was trained on imagenet with exatly this pre processing
    #preprocessing included like 1) resize/rescale pixel values 2) Reorder color channel (RGB->BGR) 3)subtract imagenet mean values
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input, 
    #below are the process of data augmentationa
    rotation_range=20, #randomly rotate images by +-20 degrees
    width_shift_range=0.2, #shift image horizontally upto 20% of its width
    height_shift_range=0.2, #shift image vertically upto 20% o its width
    horizontal_flip=True, #randomly flip some images left-rgith
    zoom_range=0.2, #randomly zooms in image by 20%
    validation_split=0.2 #tells generator when i give you a dataet please reserve 20% of it for validation
    #augmentations stimulates more training data, it makes the model ributs to slight rotations, shifts, zooms morrored views
    #hlep reducing overfitting and improgve generalization to real images
)

'''same preprcessing as train, no augmenataion, test data should represent real data not artificially modified
so train preprocessing+augmentation, test=preprocessing only'''

''' the resnet pre processing only subtracts imageNet mean, also it keras.resnet doesnt change to bgr also resnet preprocessing doesnt divide 
by 250 or standard deviation

original RGB: [180,140,120] , after preprocessing [56.32,23.22,16.06]

Why This Preprocessing?
Mean-centered data:

ResNet was trained on ImageNet with these exact mean values
Centering around 0 helps with gradient flow and convergence
Makes learning easier because features have consistent distribution

What the model interprets:

Positive values: Pixel is brighter than ImageNet average for that channel
Negative values: Pixel is darker than ImageNet average
Near zero: Pixel is close to ImageNet average'''
test_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input
)


train_images = train_gen.flow_from_dataframe(
    train_df, x_col='Filepath', y_col='Label', #train_df : a pandas data frame that has at least a filepath and a lable
    #x_col='Filepath' user file paths in this column as input images
    #use this column as labels
    target_size=(224, 224), batch_size=32, class_mode='categorical',
    #every image is resized to 224*224 to match resnet50 pixels input
    #the generator yields batches of 32 images at time
    shuffle=True, seed=42, subset='training'
    #shuffle data so model doesnt see images in same order forever
    #makes the shuffling and splitting reporducible
)
val_images = train_gen.flow_from_dataframe(
    train_df, x_col='Filepath', y_col='Label',
    target_size=(224, 224), batch_size=32, class_mode='categorical',
    shuffle=True, seed=42, subset='validation'
    #same as train just subset is validation, now we are using 20% validation split of train_df
)
test_images = test_gen.flow_from_dataframe(
    test_df, x_col='Filepath', y_col='Label',
    target_size=(224, 224), batch_size=32, class_mode='categorical',
    shuffle=False
    '''test_df
→ DataFrame for held-out test data (30% from earlier split).

x_col='Filepath', y_col='Label', target_size=(224, 224), batch_size=32, class_mode='categorical'
→ Same ideas as before.

shuffle=False
→ Very important for test data.

When you later do model.evaluate or model.predict on test_images, keeping the original order means:

the i-th prediction corresponds to the i-th row in test_df.

This is crucial if you want to make a confusion matrix or per-sample analysis.

test_gen (no augmentation)
→ Only applies the ResNet preprocessing, no random changes.'''
)

# 4. MODEL SETUP & FINE-TUNING
with tf.device(DEVICE): #everything inside this block should run on this device '/GPU0' or '/CPU:0'
    base_model = tf.keras.applications.ResNet50( #this loads a pretrained ResNet50 network
        input_shape=(224,224,3), #moel expects images of size 224*224 with 3 coloro channel
        include_top=False, #dont load the original imagenet classfication head (the part that output 1000 classes)
        weights='imagenet', #initialize the model with weight already trained on imAGEnET
        pooling='avg'
        '''pooling='avg'
→ Adds a global average pooling layer on top of the convolutional features.

Takes each feature map and averages spatially to get one number.

Result is a 1D vector instead of a 2D feature map → easier to connect to Dense layers.'''
    )
    base_model.trainable = True #first you allow all layers to be trainable
    FT_START = len(base_model.layers) - 30
    for l in base_model.layers[:FT_START]: #loops over all layers from index 0 upto FT_start-1
        l.trainable = False #their weights will not be updated during backprop, speeds up training

    '''len(base_model.layers) = total number of layers in ResNet50 (e.g., ~175-ish).

len(...) - 30 = index such that:

Earlier layers (0 to FT_START-1) will be frozen.

Last 30 layers (FT_START to end) will stay trainable.
Only fine-tune the last ~30 layers of ResNet; keep the rest frozen
early layers learn very generic features(edges,textures) that are useful for alsmot any image task, later layers are more task-specific'''


'''this part does not freeze/unfreeze anything, it just builds a new classification head on top of base_model.output'''
#all these dense layers anre new and fully trainable by default
    # Top layers
    x = tf.keras.layers.Dense(256, activation='relu')(base_model.output)
#a dense fully connected layer with 256 neurons, input: the feature vecotry coming from resnet 50 after global average pooling
#activation=relu introduces non-lineariyr, this layer learns rich representations specific to your food dase
    x = tf.keras.layers.Dropout(0.3)(x)
#dropout with rate 0.3, during training 30% of neurons are randomly turned off, reduce overfitting, learn more robust features
    x = tf.keras.layers.Dense(128, activation='relu')(x)
#another dense layers with 128 neurons , further compresses and transoforms the features
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
#final classification layerm, num_classes(units one food per cateogry), softmax : outputs a probability distribution over classes
#values between 0 and 1 , sum of all outputs=1
    model = tf.keras.Model(base_model.input, outputs)
#connects everything into a single keras model
#input: images for ResNet50
#output: your softmax classification layer
#so model=ResNet50 feature extractor + your custom dense head

    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4) 
#adam very popular optimizer, 1e-4 very small learning rate which is good for fine-turning pre trained model
#you dont want to destroy the learned imageNet features with hige updates
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
#standard loss for multiclass classification with one hot labels
    print(model.summary())

# 5. TRAIN
history = model.fit(
    train_images, validation_data=val_images, epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    ]
)
'''model.fit(...)

train_images: generator for training data (with augmentation).

validation_data=val_images: generator for validation data.

epochs=20: maximum of 20 epochs of training.'''

# 6. EVALUATE AND SAVE
results = model.evaluate(test_images, verbose=0)
print(f"Test Accuracy (ResNet50 fine-tuned): {results[1]*100:.2f}%")
model.save("food_resnet50_finetuned.h5")


GPUs available: []
⚠️ No GPU detected. Training will run on CPU.
Found 101 classes
Found 11312 validated image filenames belonging to 101 classes.
Found 2828 validated image filenames belonging to 101 classes.
Found 6060 validated image filenames belonging to 101 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,158,181 (92.16 MB)

 Trainable params: 15,020,645 (57.30 MB)

 Non-trainable params: 9,137,536 (34.86 MB)

None
Epoch 1/20
354/354 ━━━━━━━━━━━━━━━━━━━━ 193s 535ms/step - accuracy: 0.0911 - loss: 4.2235 - val_accuracy: 0.2988 - val_loss: 3.2326
Epoch 2/20
354/354 ━━━━━━━━━━━━━━━━━━━━ 194s 547ms/step - accuracy: 0.3000 - loss: 3.1141 - val_accuracy: 0.4409 - val_loss: 2.5112
Epoch 3/20
354/354 ━━━━━━━━━━━━━━━━━━━━ 197s 557ms/step - accuracy: 0.4325 - loss: 2.4054 - val_accuracy: 0.5138 - val_loss: 2.1820
Epoch 4/20
354/354 ━━━━━━━━━━━━━━━━━━━━ 197s 557ms/step - accuracy: 0.5274 - loss: 1.9813 - val_accuracy: 0.5446 - val_loss: 1.9988
Epoch 5/20
354/354 ━━━━━━━━━━━━━━━━━━━━ 191s 539ms/step - accuracy: 0.5865 - loss: 1.6713 - val_accuracy: 0.5679 - val_loss: 1.9079
Epoch 6/20
354/354 ━━━━━━━━━━━━━━━━━━━━ 188s 532ms/step - accuracy: 0.6482 - loss: 1.4111 - val_accuracy: 0.5767 - val_loss: 1.8870
Epoch 7/20
354/354 ━━━━━━━━━━━━━━━━━━━━ 188s 532ms/step - accuracy: 0.6923 - loss: 1.2309 - val_accuracy: 0.5718 - val_loss: 1.8680
Epoch 8/20
354/354 ━━━━━━━━━━━━━━━━━━━━ 189s 533ms/step - accuracy: 0.7

Test Accuracy (ResNet50 fine-tuned): 58.45%


In [8]:
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf

# 1. GPU SETUP
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        DEVICE = '/GPU:0'
        print("✅ Using device:", DEVICE)
    except RuntimeError as e:
        print("GPU config error:", e)
        DEVICE = '/CPU:0'
else:
    print("⚠️ No GPU detected. Training will run on CPU.")
    DEVICE = '/CPU:0'

# 2. DATA PREP
image_dir = Path(r'C:\Users\dkushwaha\Downloads\archive\images')  # <- Replace with your folder
filepaths = list(image_dir.glob('**/*.jpg'))
labels = [os.path.split(os.path.split(str(x))[0])[1] for x in filepaths]
df = pd.DataFrame({'Filepath': list(map(str, filepaths)), 'Label': labels})

num_classes = df['Label'].nunique()
print(f"Found {num_classes} classes")

category_samples = []
for category in df['Label'].unique():
    cat_slice = df[df['Label'] == category]
    n_samples = min(200, len(cat_slice))
    if n_samples > 0:
        category_samples.append(cat_slice.sample(n_samples, random_state=1))
df_balanced = pd.concat(category_samples).sample(frac=1.0, random_state=1).reset_index(drop=True)

train_df, test_df = train_test_split(df_balanced, train_size=0.7, shuffle=True, random_state=42)

# 3. DATA GENERATORS
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2
)
test_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input
)
train_images = train_gen.flow_from_dataframe(
    train_df, x_col='Filepath', y_col='Label',
    target_size=(224, 224), batch_size=32, class_mode='categorical',
    shuffle=True, seed=42, subset='training'
)
val_images = train_gen.flow_from_dataframe(
    train_df, x_col='Filepath', y_col='Label',
    target_size=(224, 224), batch_size=32, class_mode='categorical',
    shuffle=True, seed=42, subset='validation'
)
test_images = test_gen.flow_from_dataframe(
    test_df, x_col='Filepath', y_col='Label',
    target_size=(224, 224), batch_size=32, class_mode='categorical',
    shuffle=False
)

# 4. MODEL SETUP & FINE-TUNING
with tf.device(DEVICE):
    base_model = tf.keras.applications.ResNet50(
        input_shape=(224,224,3),
        include_top=False,
        weights='imagenet',
        pooling='avg'
    )
    base_model.trainable = True
    FT_START = len(base_model.layers) - 30
    for l in base_model.layers[:FT_START]:
        l.trainable = False

    # Top layers
    x = tf.keras.layers.Dense(256, activation='relu')(base_model.output)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(base_model.input, outputs)

    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    print(model.summary())

# 5. TRAIN
history = model.fit(
    train_images, validation_data=val_images, epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    ]
)

# 6. EVALUATE AND SAVE
results = model.evaluate(test_images, verbose=0)
print(f"Test Accuracy (ResNet50 fine-tuned): {results[1]*100:.2f}%")
model.save("food_resnet50_finetuned.h5")


GPUs available: []
⚠️ No GPU detected. Training will run on CPU.
Found 72 classes
Found 8064 validated image filenames belonging to 72 classes.
Found 2016 validated image filenames belonging to 72 classes.
Found 4320 validated image filenames belonging to 72 classes.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,154,440 (92.14 MB)

 Trainable params: 15,016,904 (57.28 MB)

 Non-trainable params: 9,137,536 (34.86 MB)

None
Epoch 1/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 152s 589ms/step - accuracy: 0.1352 - loss: 3.7528 - val_accuracy: 0.3914 - val_loss: 2.6069
Epoch 2/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 143s 567ms/step - accuracy: 0.3952 - loss: 2.5543 - val_accuracy: 0.5322 - val_loss: 1.9862
Epoch 3/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 140s 554ms/step - accuracy: 0.5185 - loss: 1.9543 - val_accuracy: 0.5893 - val_loss: 1.7596
Epoch 4/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 140s 557ms/step - accuracy: 0.6120 - loss: 1.5433 - val_accuracy: 0.5952 - val_loss: 1.6852
Epoch 5/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 140s 556ms/step - accuracy: 0.6777 - loss: 1.2598 - val_accuracy: 0.6017 - val_loss: 1.6672
Epoch 6/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 140s 556ms/step - accuracy: 0.7236 - loss: 1.0585 - val_accuracy: 0.6111 - val_loss: 1.7113
Epoch 7/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 219s 870ms/step - accuracy: 0.7656 - loss: 0.8906 - val_accuracy: 0.6319 - val_loss: 1.6866
Epoch 8/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 154s 610ms/step - accuracy: 0.8

Test Accuracy (ResNet50 fine-tuned): 63.87%


In [10]:
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf

# 1. GPU SETUP
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        DEVICE = '/GPU:0'
        print("✅ Using device:", DEVICE)
    except RuntimeError as e:
        print("GPU config error:", e)
        DEVICE = '/CPU:0'
else:
    print("⚠️ No GPU detected. Training will run on CPU.")
    DEVICE = '/CPU:0'

# 2. DATA PREP
image_dir = Path(r'C:\Users\dkushwaha\Downloads\archive\images')  # <- Replace with your folder
filepaths = list(image_dir.glob('**/*.jpg'))
labels = [os.path.split(os.path.split(str(x))[0])[1] for x in filepaths]
df = pd.DataFrame({'Filepath': list(map(str, filepaths)), 'Label': labels})

num_classes = df['Label'].nunique()
print(f"Found {num_classes} classes")

category_samples = []
for category in df['Label'].unique():
    cat_slice = df[df['Label'] == category]
    n_samples = min(200, len(cat_slice))
    if n_samples > 0:
        category_samples.append(cat_slice.sample(n_samples, random_state=1))
df_balanced = pd.concat(category_samples).sample(frac=1.0, random_state=1).reset_index(drop=True)

train_df, test_df = train_test_split(df_balanced, train_size=0.7, shuffle=True, random_state=42)

# 3. DATA GENERATORS
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2
)
test_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input
)
train_images = train_gen.flow_from_dataframe(
    train_df, x_col='Filepath', y_col='Label',
    target_size=(224, 224), batch_size=32, class_mode='categorical',
    shuffle=True, seed=42, subset='training'
)
val_images = train_gen.flow_from_dataframe(
    train_df, x_col='Filepath', y_col='Label',
    target_size=(224, 224), batch_size=32, class_mode='categorical',
    shuffle=True, seed=42, subset='validation'
)
test_images = test_gen.flow_from_dataframe(
    test_df, x_col='Filepath', y_col='Label',
    target_size=(224, 224), batch_size=32, class_mode='categorical',
    shuffle=False
)

# 4. MODEL SETUP & FINE-TUNING
with tf.device(DEVICE):
    base_model = tf.keras.applications.ResNet50(
        input_shape=(224,224,3),
        include_top=False,
        weights='imagenet',
        pooling='avg'
    )
    base_model.trainable = True
    FT_START = len(base_model.layers) - 30
    for l in base_model.layers[:FT_START]:
        l.trainable = False

    # Top layers
    x = tf.keras.layers.Dense(256, activation='relu')(base_model.output)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(base_model.input, outputs)

    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    print(model.summary())

# 5. TRAIN
history = model.fit(
    train_images, validation_data=val_images, epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    ]
)

# 6. EVALUATE AND SAVE
results = model.evaluate(test_images, verbose=0)
print(f"Test Accuracy (ResNet50 fine-tuned): {results[1]*100:.2f}%")
model.save("food_resnet50_finetuned_52_class.h5")


GPUs available: []
⚠️ No GPU detected. Training will run on CPU.
Found 52 classes
Found 5824 validated image filenames belonging to 52 classes.
Found 1455 validated image filenames belonging to 52 classes.
Found 3121 validated image filenames belonging to 52 classes.


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_3[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,151,860 (92.13 MB)

 Trainable params: 15,014,324 (57.28 MB)

 Non-trainable params: 9,137,536 (34.86 MB)

None
Epoch 1/20
182/182 ━━━━━━━━━━━━━━━━━━━━ 154s 822ms/step - accuracy: 0.1636 - loss: 3.4266 - val_accuracy: 0.4529 - val_loss: 2.2315
Epoch 2/20
182/182 ━━━━━━━━━━━━━━━━━━━━ 150s 827ms/step - accuracy: 0.4578 - loss: 2.1743 - val_accuracy: 0.5842 - val_loss: 1.7184
Epoch 3/20
182/182 ━━━━━━━━━━━━━━━━━━━━ 174s 957ms/step - accuracy: 0.5943 - loss: 1.5907 - val_accuracy: 0.6330 - val_loss: 1.4648
Epoch 4/20
182/182 ━━━━━━━━━━━━━━━━━━━━ 189s 1s/step - accuracy: 0.6908 - loss: 1.2036 - val_accuracy: 0.6323 - val_loss: 1.4522
Epoch 5/20
182/182 ━━━━━━━━━━━━━━━━━━━━ 211s 1s/step - accuracy: 0.7455 - loss: 0.9578 - val_accuracy: 0.6371 - val_loss: 1.4429
Epoch 6/20
182/182 ━━━━━━━━━━━━━━━━━━━━ 210s 869ms/step - accuracy: 0.7946 - loss: 0.7832 - val_accuracy: 0.6474 - val_loss: 1.4661
Epoch 7/20
182/182 ━━━━━━━━━━━━━━━━━━━━ 181s 995ms/step - accuracy: 0.8295 - loss: 0.6173 - val_accuracy: 0.6467 - val_loss: 1.4828
Epoch 8/20
182/182 ━━━━━━━━━━━━━━━━━━━━ 201s 1s/step - accuracy: 0.8623 - los

Test Accuracy (ResNet50 fine-tuned): 67.16%


In [ ]:
# ============================================================================
# SECTION 1: IMPORTS
# ============================================================================
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf

# os: For file path operations
# Path: Modern way to handle file paths (cleaner than os.path)
# pandas: To organize image paths and labels in a DataFrame (easier to manipulate)
# train_test_split: Split data into training and testing sets
# tensorflow: Deep learning framework

# ============================================================================
# SECTION 2: GPU CONFIGURATION
# ============================================================================
# RATIONALE: Deep learning is MUCH faster on GPU. We need to:
# 1. Check if GPU is available
# 2. Configure it properly to avoid memory errors
# 3. Fall back to CPU if no GPU

gpus = tf.config.list_physical_devices('GPU')
# List all available GPUs on your system
# Returns empty list [] if no GPU detected

print("GPUs available:", gpus)

if gpus:
    # If at least one GPU is found
    try:
        for gpu in gpus:
            # CRITICAL: Set memory growth = True
            tf.config.experimental.set_memory_growth(gpu, True)
            # WHY: By default, TensorFlow allocates ALL GPU memory at once
            # This causes crashes if multiple programs need GPU
            # memory_growth=True means: allocate memory as needed (safer)
        
        DEVICE = '/GPU:0'
        # Use the first GPU (GPU:0)
        print("✅ Using device:", DEVICE)
    
    except RuntimeError as e:
        # If GPU configuration fails (driver issues, etc.)
        print("GPU config error:", e)
        DEVICE = '/CPU:0'
        # Fall back to CPU
else:
    # No GPU detected
    print("⚠️ No GPU detected. Training will run on CPU.")
    DEVICE = '/CPU:0'
    # Use CPU instead (much slower, but works)

# ============================================================================
# SECTION 3: DATA PREPARATION
# ============================================================================
# RATIONALE: We need to organize our images and their labels
# Images are typically stored in folders like:
# images/pizza/image1.jpg
# images/burger/image2.jpg
# We need to extract the labels (folder names) and create a dataset

image_dir = Path(r'C:\Users\dkushwaha\Downloads\archive\images')
# Path to your dataset folder
# Path() creates a Path object for easier file operations

filepaths = list(image_dir.glob('**/*.jpg'))
# glob('**/*.jpg') finds ALL .jpg files in ALL subdirectories
# '**' means "search recursively in all folders"
# Returns a list of Path objects pointing to each image

labels = [os.path.split(os.path.split(str(x))[0])[1] for x in filepaths]
# Extract the label (folder name) from each file path
# Example: 'images/pizza/img001.jpg' → 'pizza'
# BREAKDOWN:
# 1. str(x) → Convert Path to string: 'images/pizza/img001.jpg'
# 2. os.path.split(str(x)) → ('images/pizza', 'img001.jpg')
# 3. [0] → Take the directory part: 'images/pizza'
# 4. os.path.split(...)[0] → ('images', 'pizza')
# 5. [1] → Take the folder name: 'pizza'

df = pd.DataFrame({'Filepath': list(map(str, filepaths)), 'Label': labels})
# Create a DataFrame (like an Excel table):
# | Filepath                      | Label  |
# |-------------------------------|--------|
# | images/pizza/img001.jpg       | pizza  |
# | images/burger/img002.jpg      | burger |
# WHY DataFrame: Easy to shuffle, split, filter, and pass to Keras

num_classes = df['Label'].nunique()
# Count unique labels (e.g., pizza, burger, sushi = 3 classes)
print(f"Found {num_classes} classes")

# ============================================================================
# SECTION 4: BALANCED SAMPLING
# ============================================================================
# RATIONALE: Imbalanced datasets cause problems!
# If you have 1000 pizza images but only 50 sushi images,
# the model will be biased toward pizza
# SOLUTION: Take equal samples from each class (up to 200 per class)

category_samples = []
# Empty list to store balanced samples

for category in df['Label'].unique():
    # Loop through each unique label (pizza, burger, sushi, etc.)
    
    cat_slice = df[df['Label'] == category]
    # Filter DataFrame to get only rows for this category
    # Example: all rows where Label == 'pizza'
    
    n_samples = min(200, len(cat_slice))
    # Take 200 samples, OR less if category has fewer images
    # Example: if pizza has 500 images → take 200
    # if sushi has 50 images → take 50
    
    if n_samples > 0:
        # Make sure category has at least 1 image
        category_samples.append(cat_slice.sample(n_samples, random_state=1))
        # Randomly sample n_samples from this category
        # random_state=1 ensures reproducibility (same samples every run)

df_balanced = pd.concat(category_samples).sample(frac=1.0, random_state=1).reset_index(drop=True)
# STEP 1: concat() combines all category samples into one DataFrame
# STEP 2: .sample(frac=1.0) shuffles the entire dataset (frac=1.0 means 100% of data)
# STEP 3: .reset_index(drop=True) resets row numbers to 0, 1, 2, 3...
# WHY SHUFFLE: So images aren't grouped by class (all pizzas, then all burgers)

# ============================================================================
# SECTION 5: TRAIN-TEST SPLIT
# ============================================================================
# RATIONALE: We need separate data to:
# 1. Train the model (train set)
# 2. Evaluate how well it works on unseen data (test set)

train_df, test_df = train_test_split(df_balanced, train_size=0.7, shuffle=True, random_state=42)
# Split dataset: 70% for training, 30% for testing
# shuffle=True: Randomly mix data before splitting
# random_state=42: Ensures same split every time you run the code
# WHY 70/30 SPLIT: Common standard. Enough data to train, enough to test.

# ============================================================================
# SECTION 6: DATA GENERATORS (IMAGE AUGMENTATION)
# ============================================================================
# RATIONALE: We have limited data (200 images per class)
# SOLUTION: Create "artificial" variations of images during training
# This makes the model more robust and prevents overfitting

train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    # ImageDataGenerator: Loads images in batches and applies transformations
    
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    # CRITICAL: Normalize pixel values for ResNet50
    # ResNet50 expects specific preprocessing (converts RGB to BGR, normalizes)
    # WHY: ResNet was trained with this preprocessing, so we must match it
    
    rotation_range=20,
    # Randomly rotate images up to ±20 degrees
    # WHY: Food can be photographed from any angle
    # Example: pizza tilted 15 degrees is still pizza
    
    width_shift_range=0.2,
    # Randomly shift image left/right by up to 20% of width
    # WHY: Object might not always be centered
    
    height_shift_range=0.2,
    # Randomly shift image up/down by up to 20% of height
    
    horizontal_flip=True,
    # Randomly flip image horizontally (mirror)
    # WHY: A burger looks the same from left or right
    # NOTE: Don't use for text or faces (would look wrong!)
    
    zoom_range=0.2,
    # Randomly zoom in/out by up to 20%
    # WHY: Photos can be taken closer or farther
    
    validation_split=0.2
    # Reserve 20% of training data for validation
    # VALIDATION: Check model performance during training
    # WHY: Helps detect overfitting early
)

test_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input
    # For test set: ONLY preprocess, NO augmentation
    # WHY: We want to evaluate on real, unmodified images
)

# ============================================================================
# SECTION 7: CREATE DATA FLOWS
# ============================================================================
# RATIONALE: Load images in batches (not all at once)
# Loading 10,000 images into RAM would crash your computer!
# Instead: load 32 images at a time

train_images = train_gen.flow_from_dataframe(
    train_df,
    # Source DataFrame with image paths and labels
    
    x_col='Filepath',
    # Column containing image paths
    
    y_col='Label',
    # Column containing labels
    
    target_size=(224, 224),
    # Resize all images to 224x224 pixels
    # WHY: ResNet50 requires fixed size input
    # All images must have same dimensions
    
    batch_size=32,
    # Load 32 images at a time
    # WHY: Training in batches is faster and uses less memory
    # 32 is a common default (powers of 2 work well with GPUs)
    
    class_mode='categorical',
    # One-hot encode labels
    # Example: 'pizza' becomes [1, 0, 0, 0, ...] if pizza is class 0
    # WHY: Neural networks need numerical labels
    
    shuffle=True,
    # Shuffle images every epoch
    # WHY: Prevents model from learning order patterns
    
    seed=42,
    # Random seed for reproducibility
    
    subset='training'
    # Use 80% of train_df (remember validation_split=0.2)
)

val_images = train_gen.flow_from_dataframe(
    train_df,
    # SAME train_df as above
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True,
    seed=42,
    subset='validation'
    # Use the OTHER 20% of train_df for validation
    # WHY: Monitor performance during training without touching test set
)

test_images = test_gen.flow_from_dataframe(
    test_df,
    # Use test_df (the 30% we split earlier)
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
    # DON'T shuffle test set
    # WHY: Want to evaluate in consistent order, easier to debug predictions
)

# ============================================================================
# SECTION 8: MODEL ARCHITECTURE (TRANSFER LEARNING)
# ============================================================================
# RATIONALE: Training a CNN from scratch needs:
# - Millions of images
# - Days/weeks of training
# - Expensive GPUs
# SOLUTION: Use ResNet50 pre-trained on ImageNet, adapt it for food

with tf.device(DEVICE):
    # Force all operations to run on specified device (GPU or CPU)
    
    base_model = tf.keras.applications.ResNet50(
        # Load ResNet50 architecture
        # ResNet50 = 50-layer deep network with skip connections
        
        input_shape=(224,224,3),
        # Input: 224x224 pixel RGB images (3 channels)
        
        include_top=False,
        # CRITICAL: Remove ResNet's original classification layers
        # WHY: ResNet was trained for 1000 ImageNet classes
        # We need 52 food classes instead!
        
        weights='imagenet',
        # Load pre-trained weights from ImageNet
        # These weights learned to recognize edges, shapes, textures
        # WHY: These features are useful for ANY image task
        
        pooling='avg'
        # Add Global Average Pooling after last conv layer
        # Converts [7, 7, 2048] feature maps → [2048] vector
        # WHY: Need fixed-size vector for dense layers
    )
    
    # ========================================================================
    # FINE-TUNING STRATEGY
    # ========================================================================
    base_model.trainable = True
    # Allow ResNet weights to be updated during training
    
    FT_START = len(base_model.layers) - 30
    # Calculate which layer to start fine-tuning from
    # ResNet50 has ~175 layers total
    # FT_START = 175 - 30 = 145
    
    for l in base_model.layers[:FT_START]:
        l.trainable = False
    # Freeze first 145 layers, only train last 30
    # WHY:
    # - Early layers learn generic features (edges, colors)
    # - These are universal, don't need updating
    # - Late layers learn specific features (objects, parts)
    # - These need adaptation for food images
    # BENEFIT: Faster training, less overfitting

    # ========================================================================
    # CUSTOM CLASSIFICATION HEAD
    # ========================================================================
    # Build layers on top of ResNet50 base
    
    x = tf.keras.layers.Dense(256, activation='relu')(base_model.output)
    # Dense layer: 2048 input → 256 output neurons
    # ReLU activation: f(x) = max(0, x)
    # RATIONALE:
    # - Compress 2048 ResNet features into 256
    # - Learn combinations of features specific to food
    # - ReLU adds non-linearity (without it, network is just linear algebra)
    
    x = tf.keras.layers.Dropout(0.3)(x)
    # Randomly set 30% of neurons to 0 during training
    # RATIONALE:
    # - Prevents overfitting (model relying too much on specific neurons)
    # - Forces network to learn redundant representations
    # - Like ensemble learning within one network
    # - NOT applied during testing (all neurons active)
    
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    # Dense layer: 256 → 128 neurons
    # RATIONALE:
    # - Further compress and abstract features
    # - Creates an information bottleneck
    # - Forces network to keep only most important info
    
    x = tf.keras.layers.Dropout(0.2)(x)
    # 20% dropout (less aggressive than first dropout)
    # RATIONALE: Later layers = more specific features, less dropout needed
    
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    # Final layer: 128 → num_classes (52 for your food dataset)
    # Softmax activation: converts to probabilities
    # Example output: [0.8, 0.1, 0.05, 0.03, ...] sums to 1.0
    # Highest value = predicted class
    # RATIONALE: Multi-class classification requires softmax
    
    model = tf.keras.Model(base_model.input, outputs)
    # Create complete model: ResNet50 input → your custom output
    # Model structure:
    # Input (224,224,3) → ResNet50 (frozen 145 layers + trainable 30)
    #   → Dense(256) → Dropout(0.3) → Dense(128) → Dropout(0.2)
    #   → Dense(52) → Softmax

    # ========================================================================
    # MODEL COMPILATION
    # ========================================================================
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    # Adam optimizer with learning rate = 0.0001
    # ADAM = Adaptive Moment Estimation
    # RATIONALE:
    # - Adapts learning rate for each parameter
    # - Works well for most problems
    # - 1e-4 is LOW learning rate (good for fine-tuning)
    # WHY LOW: Pre-trained weights are already good, small adjustments only
    
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        # Loss function for multi-class classification
        # Measures "how wrong" predictions are
        # Lower loss = better predictions
        
        metrics=['accuracy']
        # Track accuracy during training
        # Accuracy = % of correct predictions
    )
    
    print(model.summary())
    # Show model architecture: layers, parameters, connections

# ============================================================================
# SECTION 9: TRAINING
# ============================================================================
# RATIONALE: Now we actually train the model to recognize food classes

history = model.fit(
    train_images,
    # Training data generator (80% of 70% = 56% of total data)
    
    validation_data=val_images,
    # Validation data (20% of 70% = 14% of total data)
    # Used to monitor performance during training
    # WHY: Detect if model is overfitting (training acc >> val acc)
    
    epochs=20,
    # Train for 20 complete passes through the dataset
    # Each epoch: model sees all training images once
    # WHY 20: Enough for convergence, not too much (overfitting risk)
    
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            # Watch validation loss
            
            patience=3,
            # Stop if val_loss doesn't improve for 3 consecutive epochs
            # RATIONALE: No point training if validation gets worse
            # Prevents wasting time and overfitting
            
            restore_best_weights=True
            # When stopping, restore weights from best epoch
            # WHY: Model might get worse in later epochs
        )
    ]
)
# Training process for each epoch:
# 1. Load batch of 32 images
# 2. Forward pass: make predictions
# 3. Calculate loss (how wrong)
# 4. Backward pass: calculate gradients
# 5. Update weights using Adam optimizer
# 6. Repeat for all batches
# 7. Evaluate on validation set
# 8. Check early stopping condition

# ============================================================================
# SECTION 10: EVALUATION AND SAVING
# ============================================================================

results = model.evaluate(test_images, verbose=0)
# Evaluate on test set (the 30% we never touched during training)
# Returns [loss, accuracy]
# WHY: True measure of model performance on unseen data

print(f"Test Accuracy (ResNet50 fine-tuned): {results[1]*100:.2f}%")
# Print accuracy as percentage
# Example: 0.87 → 87.00%

model.save("food_resnet50_finetuned_52_class.h5")
# Save trained model to disk
# .h5 format includes:
# - Model architecture
# - Trained weights
# - Optimizer state
# WHY: Can load later for predictions without retraining
# Usage: model = tf.keras.models.load_model("food_resnet50_finetuned_52_class.h5")